# 🚁 دموی گرافیکی پهپاد: فاصله، زاویه و تصمیم حرکت
با سه Slider، **مانع را جابه‌جا می‌کنیم، فاصله را تغییر می‌دهیم و پهپاد را می‌چرخانیم**. هم‌زمان نمای بالا، تصویر دوربین، زاویه، تانژانت و فرمان حرکت را می‌بینیم.

این شبیه‌سازی ساده‌شده و آموزشی است.

In [ ]:
import sys, subprocess, importlib.util
for package,module in [('arabic-reshaper','arabic_reshaper'),('python-bidi','bidi')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle, Polygon, Wedge
from ipywidgets import interact, FloatSlider
import arabic_reshaper
from bidi.algorithm import get_display
def fa(text): return get_display(arabic_reshaper.reshape(str(text)))
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass
plt.rcParams['figure.dpi']=110
print('آماده است؛ سلول تعاملی پایین را اجرا کن.')

## ایدهٔ ریاضی
پهپاد بین جهت نگاه خود و مانع یک مثلث می‌سازد:

$$\tan(\beta)=\frac{\text{فاصله جانبی مانع}}{\text{فاصله روبه‌جلو}}$$

زاویه مثبت یعنی مانع سمت راست دوربین است؛ زاویه منفی یعنی مانع سمت چپ است. فرمان ایمن جهت مخالف را انتخاب می‌کند.

## 🎮 Sliderها را حرکت بده
ابتدا `زاویه پهپاد` را صفر نگه دار و مانع را کاملاً چپ و راست ببر. سپس مانع را ثابت نگه دار و فقط خود پهپاد را بچرخان.

In [ ]:
IMAGE_W, IMAGE_H = 640, 360
CENTER, FOCAL = IMAGE_W/2, 400
REAL_WIDTH, HALF_FOV = .20, 35

def wrap(a): return (a+180)%360-180

def drone(ax,heading):
    h=np.radians(heading); f=np.array([np.sin(h),np.cos(h)]); r=np.array([np.cos(h),-np.sin(h)])
    ax.add_patch(Polygon([.34*f,-.22*f+.22*r,-.22*f-.22*r],color='#2387c9',ec='#17324d',lw=2,zorder=6))
    for q in [.19*r,-.19*r]: ax.add_patch(Circle(q,.11,fill=False,ec='#17324d',lw=2,zorder=6))
    ax.arrow(0,0,f[0]*.75,f[1]*.75,width=.018,head_width=.14,color='#2387c9',zorder=5)

def demo(obstacle_x=1.5, forward_distance=3.0, drone_heading=0.0):
    obs=np.array([obstacle_x,forward_distance]); distance=float(np.linalg.norm(obs))
    world=np.degrees(np.arctan2(obstacle_x,forward_distance)); beta=wrap(world-drone_heading)
    tangent=np.tan(np.radians(beta)); camera_x=CENTER+FOCAL*tangent
    px_width=FOCAL*REAL_WIDTH/max(distance,.1); visible=abs(beta)<=HALF_FOV
    if not visible: decision,arrow,color='مانع خارج از دید دوربین است','×','#63758a'
    elif beta>2: decision,arrow,color='مانع راست است → حرکت به چپ','←','#ec5366'
    elif beta<-2: decision,arrow,color='مانع چپ است → حرکت به راست','→','#2387c9'
    else: decision,arrow,color='مانع تقریباً روبه‌رو است','↔','#f2a93b'

    fig=plt.figure(figsize=(14,6.2),facecolor='#f7fbff'); gs=fig.add_gridspec(1,2,width_ratios=[1.15,1],wspace=.15)
    ax=fig.add_subplot(gs[0,0]); cam=fig.add_subplot(gs[0,1])
    ax.set(xlim=(-3.2,3.2),ylim=(-.8,5.4),xlabel=fa('چپ  ←   موقعیت جانبی   →  راست'),ylabel=fa('فاصله روبه‌جلو'))
    ax.set_aspect('equal');ax.set_facecolor('white');ax.grid(alpha=.15);ax.set_title(fa('نمای بالا: پهپاد و مانع'),weight='bold')
    ax.add_patch(Wedge((0,0),5,90-(drone_heading+HALF_FOV),90-(drone_heading-HALF_FOV),color='#bde7f7',alpha=.30))
    drone(ax,drone_heading);ax.plot([0,obstacle_x],[0,forward_distance],'--',lw=2.5,color='#ec5366')
    ax.add_patch(Circle(obs,.24,color='#ec5366',ec='#9f2939',lw=2,zorder=7));ax.text(obstacle_x,forward_distance+.35,fa('مانع'),ha='center',weight='bold',color='#9f2939')
    ax.plot([0,0],[0,forward_distance],':',lw=2,color='#32a67c');ax.plot([0,obstacle_x],[forward_distance,forward_distance],':',lw=2,color='#f2a93b')
    ax.text(obstacle_x/2,forward_distance+.08,fa(f'جانبی = {obstacle_x:+.1f} m'),ha='center',color='#9a6a00')
    ax.text(.08,forward_distance/2,fa(f'روبه‌جلو = {forward_distance:.1f} m'),rotation=90,va='center',color='#23805f')

    cam.set(xlim=(0,IMAGE_W),ylim=(IMAGE_H,0),xlabel=fa('پیکسل افقی تصویر'));cam.set_facecolor('#dff3fb');cam.set_yticks([])
    cam.axvline(CENTER,color='#f2a93b',ls='--',lw=3);cam.text(CENTER,25,fa('مرکز تصویر'),ha='center',weight='bold',color='#9a6a00')
    cam.add_patch(Rectangle((0,275),IMAGE_W,85,color='#8dcf91',alpha=.75));cam.set_title(fa('آنچه دوربین می‌بیند'),weight='bold')
    if visible:
        dw=np.clip(px_width*2.4,35,150);dh=dw*1.45;dx=np.clip(camera_x-dw/2,2,IMAGE_W-dw-2)
        cam.add_patch(Rectangle((dx,255-dh),dw,dh,color='#ec5366',ec='#9f2939',lw=2));cam.text(dx+dw/2,255-dh/2,fa('مانع'),ha='center',va='center',color='white',weight='bold')
    else:
        side='راست' if beta>0 else 'چپ';cam.text(CENTER,175,fa(f'مانع بیرون کادر و سمت {side} است'),ha='center',weight='bold',color='#63758a')

    card=dict(boxstyle='round,pad=.55',fc='white',lw=2)
    fig.text(.10,.015,fa(f'فاصله: {distance:.2f} m'),ha='center',weight='bold',bbox={**card,'ec':'#2387c9'})
    fig.text(.29,.015,f'tan(β): {tangent:+.3f}',ha='center',weight='bold',bbox={**card,'ec':'#f2b84b'})
    fig.text(.47,.015,fa(f'زاویه β: {beta:+.1f}°'),ha='center',weight='bold',bbox={**card,'ec':'#9b6dcc'})
    fig.text(.75,.015,fa(f'{arrow}  {decision}'),ha='center',fontsize=13,weight='bold',color=color,bbox={**card,'ec':color})
    plt.subplots_adjust(bottom=.16,top=.88);plt.show()
    print(f'زاویه واقعی مانع = arctan({obstacle_x:+.1f}/{forward_distance:.1f}) = {world:+.1f}°')
    print(f'زاویه دوربین = {world:+.1f}° - زاویه پهپاد {drone_heading:+.1f}° = {beta:+.1f}°')
    print(f'tan(β) = {tangent:+.3f}  |  نتیجه: {decision}')

interact(demo,
 obstacle_x=FloatSlider(min=-2.5,max=2.5,step=.25,value=1.5,description='مانع چپ/راست',continuous_update=True),
 forward_distance=FloatSlider(min=.8,max=5,step=.2,value=3,description='فاصله',continuous_update=True),
 drone_heading=FloatSlider(min=-30,max=30,step=5,value=0,description='زاویه پهپاد',continuous_update=True));

## سه آزمایش کلاسی
1. **پهپاد ثابت، مانع متحرک:** زاویه پهپاد را صفر نگه دار و مانع را از راست به چپ ببر.  
2. **مانع ثابت، پهپاد چرخان:** مانع را ثابت نگه دار و فقط زاویه پهپاد را تغییر بده.  
3. **نزدیک‌شدن:** جای مانع را ثابت نگه دار و فاصله را کم کن؛ مانع در تصویر بزرگ‌تر می‌شود.

> سؤال: آیا جای واقعی مانع عوض شد، یا فقط زاویه نگاه پهپاد تغییر کرد؟

## ارتباط با کد مقاله
```python
angle = np.arctan((obstacle_center-image_center)/focal_length)
command = 'LEFT' if angle > 0 else 'RIGHT'
```
این محاسبه در هر فریم تکرار می‌شود؛ پس **فاصله، زاویه و فرمان حرکت همگی سری زمانی هستند.**